In [1]:
print(5)

5


In [3]:
#STEP-1

import pandas as pd
import spacy
from tqdm import tqdm
from pyabsa import AspectTermExtraction as ATEPC

# ── Load spaCy ─────────────────────────────────────────────────────────────────
# nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("en_core_web_md")

# ── Load data ──────────────────────────────────────────────────────────────────
df_raw = pd.read_csv("electric_tb_p2.csv")

df_raw = df_raw.rename(columns={"review_score": "star_rating"})

# ── Load PyABSA on CPU ─────────────────────────────────────────────────────────
extractor = ATEPC.AspectExtractor(
    checkpoint="multilingual",
    auto_device=False,
    cal_perplexity=False
)

[2026-03-27 22:19:51] (2.3.3) Downloading checkpoint:multilingual 
[2026-03-27 22:19:51] (2.3.3) Notice: The pretrained model are used for testing, it is recommended to train the model on your own custom datasets


Find zipped checkpoint: ./checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT/fast_lcf_atepc_Multilingual_cdw_apcacc_85.1_apcf1_80.2_atef1_76.45.zip, unzipping
Done.
[2026-03-27 22:20:19] (2.3.3) If the auto-downloading failed, please download it via browser: https://huggingface.co/spaces/yangheng/PyABSA/resolve/main/checkpoints/Multilingual/ATEPC/fast_lcf_atepc_Multilingual_cdw_apcacc_85.1_apcf1_80.2_atef1_76.45.zip 
[2026-03-27 22:20:19] (2.3.3) Load aspect extractor from checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT
[2026-03-27 22:20:19] (2.3.3) config: checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT/fast_lcf_atepc.config
[2026-03-27 22:20:19] (2.3.3) state_dict: checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT/fast_lcf_atepc.state_dict
[2026-03-27 22:20:19] (2.3.3) model: None
[2026-03-27 22:20:19] (2.3.3) tokenizer: checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT/fast_lcf_atepc.tokenizer


/opt/anaconda3/envs/myenv_DSL/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[2026-03-27 22:20:20] (2.3.3) Set Model Device: cpu
[2026-03-27 22:20:20] (2.3.3) Device Name: Unknown


Some weights of the model checkpoint at microsoft/mdeberta-v3-base were not used when initializing DebertaV2Model: ['mask_predictions.classifier.weight', 'mask_predictions.classifier.bias', 'lm_predictions.lm_head.LayerNorm.weight', 'lm_predictions.lm_head.dense.bias', 'lm_predictions.lm_head.bias', 'mask_predictions.LayerNorm.weight', 'mask_predictions.dense.bias', 'mask_predictions.dense.weight', 'deberta.embeddings.word_embeddings._weight', 'lm_predictions.lm_head.LayerNorm.bias', 'mask_predictions.LayerNorm.bias', 'lm_predictions.lm_head.dense.weight']
- This IS expected if you are initializing DebertaV2Model from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaV2Model from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSeque

In [ ]:
# ── Opinion extractor via spaCy ────────────────────────────────────────────────
def extract_opinion(sentence: str, aspect: str) -> str:
    doc           = nlp(sentence)
    aspect_tokens = [t for t in doc if t.text.lower() in aspect.lower()]

    for token in aspect_tokens:
        # direct adjectival modifier: "broken bottle", "not good bag"
        for child in token.children:
            if child.dep_ == "amod" and child.pos_ in ("ADJ", "VERB"):
                neg = next((c.text for c in child.children if c.dep_ == "neg"), None)
                return f"{neg} {child.text}" if neg else child.text

        # predicate adjective: "bottle is broken", "bag is not good"
        if token.head.pos_ == "VERB":
            for s in token.head.children:
                if s.dep_ in ("acomp", "attr") and s.pos_ == "ADJ":
                    neg = next((c.text for c in s.children if c.dep_ == "neg"), None)
                    return f"{neg} {s.text}" if neg else s.text

            # past participle: "pump stopped"
            if token.head.tag_ == "VBN":
                return token.head.text

        # relative clause: "bottle that looks damaged"
        for child in token.children:
            if child.dep_ == "relcl":
                for subchild in child.children:
                    if subchild.dep_ in ("acomp", "attr") and subchild.pos_ == "ADJ":
                        neg = next((c.text for c in subchild.children if c.dep_ == "neg"), None)
                        return f"{neg} {subchild.text}" if neg else subchild.text

    # fallback: nearest adjective in sentence
    adjs = [t for t in doc if t.pos_ == "ADJ"]
    if adjs and aspect_tokens:
        return min(adjs, key=lambda t: abs(t.i - aspect_tokens[0].i)).text

    return None


# ── Main ABSA pipeline ─────────────────────────────────────────────────────────
def run_absa(df: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing reviews"):
        review_id = row["review_id"]
        text      = str(row["review_text"])
        rating    = row["star_rating"] if "star_rating" in row else None

        result = extractor.extract_aspect(
            inference_source=[text],
            pred_sentiment=True
        )

        # result is a list with one dict
        r = result[0]

        aspects    = r["aspect"]        # ['bottle', 'bag', 'pump']
        sentiments = r["sentiment"]     # ['Negative', 'Negative', 'Negative']
        sentence   = r["sentence"]      # single string, same for all aspects
        probs      = r["probs"]         # [[p1,p2,p3], ...]

        for aspect, sentiment, prob in zip(aspects, sentiments, probs):
            opinion = extract_opinion(sentence, aspect)

            records.append({
                "review_id":    review_id,
                "star_rating":  rating,
                "sentence":     sentence,
                "aspect":       aspect,
                "opinion_term": opinion,
                "sentiment":    sentiment,
                "confidence":   round(max(prob), 4)
            })

    return pd.DataFrame(records, columns=[
        "asin","review_id", "star_rating", "sentence",
        "aspect", "opinion_term", "sentiment", "confidence"
    ])

# ── Run ───────────────────────────────────────────────-------
if __name__ == "__main__":
    df_absa = run_absa(df_raw)
    print(df_absa.to_string())
    df_absa.to_csv("absa_output.csv", index=False)
    print("\n Saved to absa_output.csv")

In [ ]:
def extract_opinion(sentence: str, aspect: str) -> str:
    doc = nlp(sentence)
    aspect_tokens = [t for t in doc if t.text.lower() in aspect.lower()]

    for token in aspect_tokens:

        # 1. Direct adjectival modifier: "broken bottle"
        for child in token.children:
            if child.dep_ == "amod" and child.pos_ in ("ADJ", "VERB"):
                neg = next((c.text for c in child.children if c.dep_ == "neg"), None)
                return f"{neg} {child.text}" if neg else child.text

        # 2. Predicate adjective: "bottle is broken"
        if token.head.pos_ == "VERB":
            for s in token.head.children:
                if s.dep_ in ("acomp", "attr") and s.pos_ == "ADJ":
                    neg = next((c.text for c in s.children if c.dep_ == "neg"), None)
                    return f"{neg} {s.text}" if neg else s.text

        # 3. NEW - Subject verb opinion: "bottle leaked", "pump stopped"
        # Aspect is subject of a verb — that verb is the opinion
        if token.dep_ in ("nsubj", "nsubjpass"):
            verb = token.head
            if verb.pos_ == "VERB":
                neg = next((c.text for c in verb.children if c.dep_ == "neg"), None)
                return f"{neg} {verb.text}" if neg else verb.text

        # 4. NEW - Verb with aspect as object: "loved the camera"
        if token.dep_ in ("dobj", "pobj"):
            verb = token.head
            if verb.pos_ == "VERB":
                neg = next((c.text for c in verb.children if c.dep_ == "neg"), None)
                return f"{neg} {verb.text}" if neg else verb.text

        # 5. Past participle acting on aspect: "bottle damaged"
        if token.head.pos_ == "VERB" and token.head.tag_ == "VBN":
            return token.head.text

        # 6. Relative clause: "bottle that looks damaged"
        for child in token.children:
            if child.dep_ == "relcl":
                for subchild in child.children:
                    if subchild.dep_ in ("acomp", "attr") and subchild.pos_ == "ADJ":
                        neg = next((c.text for c in subchild.children if c.dep_ == "neg"), None)
                        return f"{neg} {subchild.text}" if neg else subchild.text

    # 7. NEW - Fallback: nearest verb or adjective to aspect
    # Try verb first since verbs carry more opinion signal than adjectives
    opinion_candidates = [
        t for t in doc
        if t.pos_ in ("VERB", "ADJ")
        and t.text.lower() not in ("is", "are", "was", "were", "be", "been", "have", "had", "do", "did")
    ]

    if opinion_candidates and aspect_tokens:
        closest = min(
            opinion_candidates,
            key=lambda t: abs(t.i - aspect_tokens[0].i)
        )
        return closest.text

    return None


# ── Main ABSA pipeline ─────────────────────────────────────────────────────────
def run_absa(df: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing reviews"):
        review_id = row["review_id"]
        text      = str(row["review_text"])
        rating    = row["star_rating"] if "star_rating" in row else None

        result = extractor.extract_aspect(
            inference_source=[text],
            pred_sentiment=True
        )

        # result is a list with one dict
        r = result[0]

        aspects    = r["aspect"]        # ['bottle', 'bag', 'pump']
        sentiments = r["sentiment"]     # ['Negative', 'Negative', 'Negative']
        sentence   = r["sentence"]      # single string, same for all aspects
        probs      = r["probs"]         # [[p1,p2,p3], ...]

        for aspect, sentiment, prob in zip(aspects, sentiments, probs):
            opinion = extract_opinion(sentence, aspect)

            records.append({
                "review_id":    review_id,
                "star_rating":  rating,
                "sentence":     sentence,
                "aspect":       aspect,
                "opinion_term": opinion,
                "sentiment":    sentiment,
                "confidence":   round(max(prob), 4)
            })

    return pd.DataFrame(records, columns=[
        "review_id", "star_rating", "sentence",
        "aspect", "opinion_term", "sentiment", "confidence"
    ])




# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_absa = run_absa(df_raw)
    print(df_absa.to_string())
    df_absa.to_csv("absa_output_electric.csv", index=False)
    print("\n✅ Saved to absa_output.csv")

In [ ]:
def get_clause(doc, aspect_token):
    # Walk up to find the clause root
    clause_tokens = set()
    
    def collect_subtree(token):
        clause_tokens.add(token)
        for child in token.children:
            # Stop at conjunctions that separate clauses
            if child.dep_ not in ("cc", "punct"):
                collect_subtree(child)
    
    # Find the clause this aspect belongs to
    # Go up to the verb governing this aspect
    head = aspect_token
    while head.dep_ not in ("ROOT", "conj") and head.head != head:
        head = head.head
    
    collect_subtree(head)
    return clause_tokens

def extract_opinion(sentence: str, aspect: str) -> str:
    doc = nlp(sentence)
    aspect_tokens = [t for t in doc if t.text.lower() in aspect.lower()]

    for token in aspect_tokens:

        # Get only tokens in the same clause as this aspect
        clause = get_clause(doc, token)

        # 1. Direct adjectival modifier: "broken bottle"
        for child in token.children:
            if child.dep_ == "amod" and child.pos_ in ("ADJ", "VERB"):
                neg = next((c.text for c in child.children if c.dep_ == "neg"), None)
                return f"{neg} {child.text}" if neg else child.text

        # 2. Predicate adjective: "bottle is broken"
        if token.head.pos_ == "VERB":
            for s in token.head.children:
                if s.dep_ in ("acomp", "attr") and s.pos_ == "ADJ":
                    neg = next((c.text for c in s.children if c.dep_ == "neg"), None)
                    return f"{neg} {s.text}" if neg else s.text

        # 3. Subject verb opinion: "bottle leaked", "pump stopped"
        if token.dep_ in ("nsubj", "nsubjpass"):
            verb = token.head
            if verb.pos_ == "VERB" and verb in clause:
                neg = next((c.text for c in verb.children if c.dep_ == "neg"), None)
                return f"{neg} {verb.text}" if neg else verb.text

        # 4. Verb with aspect as object: "loved the camera"
        if token.dep_ in ("dobj", "pobj"):
            verb = token.head
            if verb.pos_ == "VERB" and verb in clause:
                neg = next((c.text for c in verb.children if c.dep_ == "neg"), None)
                return f"{neg} {verb.text}" if neg else verb.text

        # 5. Past participle: "bottle damaged"
        if token.head.pos_ == "VERB" and token.head.tag_ == "VBN":
            return token.head.text

        # 6. Relative clause: "bottle that looks damaged"
        for child in token.children:
            if child.dep_ == "relcl":
                for subchild in child.children:
                    if subchild.dep_ in ("acomp", "attr") and subchild.pos_ == "ADJ":
                        neg = next((c.text for c in subchild.children if c.dep_ == "neg"), None)
                        return f"{neg} {subchild.text}" if neg else subchild.text

        # 7. Fallback: nearest verb or adjective WITHIN SAME CLAUSE ONLY
        opinion_candidates = [
            t for t in clause
            if t.pos_ in ("VERB", "ADJ")
            and t != token
            and t.text.lower() not in ("is", "are", "was", "were", "be", "been", 
                                        "have", "had", "do", "did", "get", "got")
        ]

        if opinion_candidates and aspect_tokens:
            closest = min(
                opinion_candidates,
                key=lambda t: abs(t.i - aspect_tokens[0].i)
            )
            return closest.text

    return None


# ── Main ABSA pipeline ─────────────────────────────────────────────────────────
def run_absa(df: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing reviews"):
        review_id = row["review_id"]
        text      = str(row["review_text"])
        rating    = row["star_rating"] if "star_rating" in row else None

        result = extractor.extract_aspect(
            inference_source=[text],
            pred_sentiment=True
        )

        # result is a list with one dict
        r = result[0]

        aspects    = r["aspect"]        # ['bottle', 'bag', 'pump']
        sentiments = r["sentiment"]     # ['Negative', 'Negative', 'Negative']
        sentence   = r["sentence"]      # single string, same for all aspects
        probs      = r["probs"]         # [[p1,p2,p3], ...]

        for aspect, sentiment, prob in zip(aspects, sentiments, probs):
            opinion = extract_opinion(sentence, aspect)

            records.append({
                "review_id":    review_id,
                "star_rating":  rating,
                "sentence":     sentence,
                "aspect":       aspect,
                "opinion_term": opinion,
                "sentiment":    sentiment,
                "confidence":   round(max(prob), 4)
            })

    return pd.DataFrame(records, columns=[
        "review_id", "star_rating", "sentence",
        "aspect", "opinion_term", "sentiment", "confidence"
    ])




# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_absa = run_absa(df_raw)
    print(df_absa.to_string())
    df_absa.to_csv("absa_output.csv", index=False)
    print("\n✅ Saved to absa_output.csv")


In [ ]:
def smart_split(text):
    """
    Try dependency-based split first.
    Fall back to regex marker split.
    Then fall back to full sentence.
    """
    # Try dependency split
    dep_clauses = split_by_dependency(text)
    if len(dep_clauses) > 1:
        return dep_clauses
    
    # Try regex marker split
    regex_clauses = split_by_context(text)
    if len(regex_clauses) > 1:
        return regex_clauses
    
    # No split found, return as is
    return [text]

USE THIS FOR ABSA STEP-2

In [5]:
import pandas as pd
import spacy
import re
from tqdm import tqdm
from pyabsa import AspectTermExtraction as ATEPC

# Load spaCy
nlp = spacy.load("en_core_web_md")

# Load data
df_raw = pd.read_csv("electric_tb_p1.csv")
df_raw = df_raw.rename(columns={"review_score": "star_rating"})

# Load PyABSA
extractor = ATEPC.AspectExtractor(
    checkpoint="multilingual",
    auto_device=False,
    cal_perplexity=False
)

# ── Contrast and causal markers ────────────────────────────────────────────────
CONTRAST_MARKERS = [
    " but ", " however ", " although ", " though ", " yet ",
    " despite ", " even though ", " nevertheless ", " still ",
    " except ", " whereas ", " while ", " unfortunately ",
    " sadly ", " surprisingly "
]

CAUSAL_MARKERS = [
    " so ", " therefore ", " hence ", " thus ", " because ",
    " since ", " as a result ", " which is why "
]

# ── Splitting functions ────────────────────────────────────────────────────────
def split_by_context(text):
    clauses = [text]
    all_markers = CONTRAST_MARKERS + CAUSAL_MARKERS
    for marker in all_markers:
        new_clauses = []
        for clause in clauses:
            parts = re.split(re.escape(marker), clause, flags=re.IGNORECASE)
            if len(parts) > 1:
                new_clauses.extend([p.strip() for p in parts if p.strip()])
            else:
                new_clauses.append(clause)
        clauses = new_clauses
    return clauses

def split_by_dependency(text):
    doc = nlp(text)
    contrast_words = {"but", "however", "although", "though", "yet",
                      "whereas", "while", "despite", "nevertheless"}
    causal_words   = {"so", "therefore", "hence", "thus", "because", "since"}

    split_points = []
    for token in doc:
        if token.dep_ == "cc" and token.text.lower() in contrast_words | causal_words:
            split_points.append(token.i)
        if token.dep_ == "mark" and token.text.lower() in contrast_words | causal_words:
            split_points.append(token.i)

    if not split_points:
        return [text]

    clauses = []
    prev = 0
    for point in split_points:
        clause_text = doc[prev:point].text.strip()
        if clause_text:
            clauses.append(clause_text)
        prev = point + 1

    last = doc[prev:].text.strip()
    if last:
        clauses.append(last)

    return clauses

def smart_split(text):
    dep_clauses = split_by_dependency(text)
    if len(dep_clauses) > 1:
        return dep_clauses
    regex_clauses = split_by_context(text)
    if len(regex_clauses) > 1:
        return regex_clauses
    return [text]

# ── Opinion extractor ──────────────────────────────────────────────────────────
def extract_opinion(sentence: str, aspect: str) -> str:
    doc = nlp(sentence)
    aspect_tokens = [t for t in doc if t.text.lower() == aspect.lower()]

    if not aspect_tokens:
        aspect_tokens = [t for t in doc if aspect.lower() in t.text.lower()]

    if not aspect_tokens:
        return None

    for token in aspect_tokens:
        token_sent = token.sent
        sent_token_set = set(token_sent)

        # 1. Direct adjectival modifier
        for child in token.children:
            if child.dep_ == "amod" and child.pos_ in ("ADJ", "VERB"):
                neg = next((c.text for c in child.children if c.dep_ == "neg"), None)
                return f"{neg} {child.text}" if neg else child.text

        # 2. Predicate adjective
        if token.head.pos_ == "VERB" and token.head in sent_token_set:
            for s in token.head.children:
                if s.dep_ in ("acomp", "attr") and s.pos_ == "ADJ":
                    neg = next((c.text for c in s.children if c.dep_ == "neg"), None)
                    return f"{neg} {s.text}" if neg else s.text

        # 3. Subject verb
        if token.dep_ in ("nsubj", "nsubjpass"):
            verb = token.head
            if verb.pos_ == "VERB" and verb in sent_token_set:
                neg = next((c.text for c in verb.children if c.dep_ == "neg"), None)
                return f"{neg} {verb.text}" if neg else verb.text

        # 4. Object verb
        if token.dep_ in ("dobj", "pobj"):
            verb = token.head
            if verb.pos_ == "VERB" and verb in sent_token_set:
                neg = next((c.text for c in verb.children if c.dep_ == "neg"), None)
                return f"{neg} {verb.text}" if neg else verb.text

        # 5. Past participle
        if token.head.tag_ == "VBN" and token.head in sent_token_set:
            return token.head.text

        # 6. Relative clause
        for child in token.children:
            if child.dep_ == "relcl":
                for subchild in child.children:
                    if subchild.dep_ in ("acomp", "attr") and subchild.pos_ == "ADJ":
                        neg = next((c.text for c in subchild.children if c.dep_ == "neg"), None)
                        return f"{neg} {subchild.text}" if neg else subchild.text

        # 7. Fallback: nearest verb or adjective in same sentence
        stopverbs = {"is", "are", "was", "were", "be", "been", "have", "had",
                     "do", "did", "get", "got", "looks", "feel", "feels"}

        opinion_candidates = [
            t for t in sent_token_set
            if t.pos_ in ("VERB", "ADJ")
            and t != token
            and t.text.lower() not in stopverbs
        ]

        if opinion_candidates:
            closest = min(
                opinion_candidates,
                key=lambda t: abs(t.i - token.i)
            )
            return closest.text

    return None

# ── Main ABSA pipeline ─────────────────────────────────────────────────────────
def run_absa(df: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing reviews"):
        review_id = row["review_id"]
        full_text = str(row["review_text"])
        rating = row["star_rating"] if "star_rating" in row else None
        asin = row["asin"] if "asin" in row.index else None  # ADD THIS


        # Step 1 - Split into sentences
        doc = nlp(full_text)
        sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

        for sentence in sentences:
            # Step 2 - Further split by context and contrast
            clauses = smart_split(sentence)

            for clause in clauses:
                if len(clause.split()) < 3:
                    continue

                try:
                    result = extractor.extract_aspect(
                        inference_source=[clause],
                        pred_sentiment=True
                    )

                    r = result[0]
                    aspects    = r["aspect"]
                    sentiments = r["sentiment"]
                    probs      = r["probs"]

                    for aspect, sentiment, prob in zip(aspects, sentiments, probs):
                        sent_doc = nlp(clause)
                        is_location = any(
                            t.text.lower() == aspect.lower() and t.dep_ == "pobj"
                            for t in sent_doc
                        )
                        if is_location:
                            continue

                        opinion = extract_opinion(clause, aspect)

                        records.append({
                            "asin" : asin,
                            "review_id":    review_id,
                            "star_rating":  rating,
                            "sentence":     clause,
                            "aspect":       aspect,
                            "opinion_term": opinion,
                            "sentiment":    sentiment,
                            "confidence":   round(max(prob), 4)
                        })

                except Exception as e:
                    print(f"  Skipping clause due to error: {e}")
                    continue

    return pd.DataFrame(records, columns=[
        "asin","review_id", "star_rating", "sentence",
        "aspect", "opinion_term", "sentiment", "confidence"
    ])

# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_absa = run_absa(df_raw)
    print(df_absa.to_string())
    df_absa.to_csv("absa_output_electric_tb_p1.csv", index=False)
    print("\n✅ Saved to absa_output_electric_tb_p1.csv")

[2026-03-27 22:52:09] (2.3.3) Downloading checkpoint:multilingual 
[2026-03-27 22:52:09] (2.3.3) Notice: The pretrained model are used for testing, it is recommended to train the model on your own custom datasets
[2026-03-27 22:52:09] (2.3.3) Checkpoint already downloaded, skip
[2026-03-27 22:52:09] (2.3.3) Load aspect extractor from checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT
[2026-03-27 22:52:09] (2.3.3) config: checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT/fast_lcf_atepc.config
[2026-03-27 22:52:09] (2.3.3) state_dict: checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT/fast_lcf_atepc.state_dict
[2026-03-27 22:52:09] (2.3.3) model: None
[2026-03-27 22:52:09] (2.3.3) tokenizer: checkpoints/ATEPC_MULTILINGUAL_CHECKPOINT/fast_lcf_atepc.tokenizer


/opt/anaconda3/envs/myenv_DSL/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[2026-03-27 22:52:09] (2.3.3) Set Model Device: cpu
[2026-03-27 22:52:09] (2.3.3) Device Name: Unknown


Some weights of the model checkpoint at microsoft/mdeberta-v3-base were not used when initializing DebertaV2Model: ['mask_predictions.classifier.weight', 'mask_predictions.classifier.bias', 'lm_predictions.lm_head.LayerNorm.weight', 'lm_predictions.lm_head.dense.bias', 'lm_predictions.lm_head.bias', 'mask_predictions.LayerNorm.weight', 'mask_predictions.dense.bias', 'mask_predictions.dense.weight', 'deberta.embeddings.word_embeddings._weight', 'lm_predictions.lm_head.LayerNorm.bias', 'mask_predictions.LayerNorm.bias', 'lm_predictions.lm_head.dense.weight']
- This IS expected if you are initializing DebertaV2Model from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaV2Model from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSeque

[2026-03-27 22:52:21] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 22:52:21] (2.3.3) Example 0: This took a few times of using to get used to the & # 34 ; <buzz:Positive Confidence:0.8268> & # 34 ; and tickle in your mouth ,
[2026-03-27 22:52:22] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 22:52:22] (2.3.3) Example 0: wow you feel like you just had a dentist ' s <cleaning:Negative Confidence:0.6818> every time you brush .
[2026-03-27 22:52:23] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03

...

[2026-03-27 23:01:37] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 23:01:37] (2.3.3) Example 0: I faced problems related to <power:Negative Confidence:0.9962> & <motor charging system:Negative Confidence:0.9969> and overall it did not meet expectations .
[2026-03-27 23:01:38] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 23:01:38] (2.3.3) Example 0: Indicator Lights & <Display:Neutral Confidence:0.9828> : The product average <performance:Neutral Confidence:0.9611> .


Processing reviews: 100%|█████████▉| 202/203 [09:18<00:01,  1.87s/it]

[2026-03-27 23:01:39] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 23:01:39] (2.3.3) Example 0: I faced problems related to <indicator lights:Negative Confidence:0.9968> & <display:Negative Confidence:0.997> and overall it did not meet expectations .
[2026-03-27 23:01:39] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 23:01:39] (2.3.3) Example 0: Noise , Feel & Sensation :
[2026-03-27 23:01:40] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 23:01:40] (2.3.3) Example 0: The product average 

Processing reviews: 100%|██████████| 203/203 [09:21<00:00,  2.76s/it]


[2026-03-27 23:01:41] (2.3.3) The results of aspect term extraction have been saved in /Users/anmolgulati/Documents/PGDBA IIMC/IIT_KGP/DSL/PROJECT/Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-27 23:01:41] (2.3.3) Example 0: I faced problems related to <noise:Negative Confidence:0.9969> , <feel:Negative Confidence:0.9968> & <sensation:Negative Confidence:0.9971> and overall it did not meet expectations .
           asin                             review_id  star_rating                                                                                                                                                                                                                                                                       sentence                         aspect   opinion_term sentiment  confidence
0    B007VXJK6E                        A1B59UOKB5II2X            5                                                                               

In [6]:
# Concatenating 2 parts of electric dataset
df1 = pd.read_csv("absa_output_electric_tb_p2.csv")
df2 = pd.read_csv("absa_output_electric_tb_p1.csv")
df_combined = pd.concat([df1, df2], ignore_index=True)

df_combined.to_csv("absa_output_electric_tb_combined.csv", index=False)

In [5]:
import pandas as pd
import numpy as np

df_absa = pd.read_csv("absa_output_electric_tb.csv")

def classify_category(sentiment, confidence):
    if sentiment == "Positive":
        if confidence >= 0.95:
            return "Excellent"
        elif confidence >= 0.75:
            return "Good"
        else:
            return "Average"
    elif sentiment == "Negative":
        if confidence >= 0.95:
            return "Worst"
        elif confidence >= 0.75:
            return "Bad"
        else:
            return "Average"
    else:
        return "Average"

df_absa['category'] = df_absa.apply(
    lambda row: classify_category(row['sentiment'], row['confidence']), axis=1
)

print(df_absa.to_string())
df_absa.to_csv("absa_output_electric_tb.csv", index=False)
# print("Saved with category column")




from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, fcluster
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

def build_review_tree(df_absa):
    """
    Takes ABSA output dataframe and builds a 3-level review tree.
    Returns nested dictionary representing the hierarchy.
    """
    
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Get unique aspects with their average sentiment and category
    aspect_summary = df_absa.groupby('aspect').agg(
        avg_confidence = ('confidence', 'mean'),
        dominant_sentiment = ('sentiment', lambda x: x.mode()[0]),
        dominant_category = ('category', lambda x: x.mode()[0]),
        mention_count = ('aspect', 'count'),
        sample_opinions = ('opinion_term', lambda x: ' '.join(x.dropna().unique()[:3]))
    ).reset_index()
    
    aspects = aspect_summary['aspect'].tolist()
    
    if len(aspects) < 3:
        print("Too few aspects to cluster")
        return {}
    
    # Embed aspects
    embeddings = st_model.encode(aspects)
    
    # Build linkage matrix
    linkage_matrix = linkage(embeddings, method='ward')
    
    # Grid search for optimal number of clusters at each level
    def find_optimal_k(embeddings, k_range):
        scores = {}
        for k in k_range:
            if k >= len(embeddings):
                continue
            labels = fcluster(linkage_matrix, t=k, criterion='maxclust')
            if len(set(labels)) < 2:
                continue
            score = silhouette_score(embeddings, labels)
            scores[k] = score
        return max(scores, key=scores.get) if scores else k_range[0]
    
    # Find optimal cuts
    optimal_top    = find_optimal_k(embeddings, range(2, 6))
    optimal_mid    = find_optimal_k(embeddings, range(optimal_top+1, 12))
    optimal_bottom = find_optimal_k(embeddings, range(optimal_mid+1, 25))
    
    print(f"Optimal cuts — Top: {optimal_top}, Mid: {optimal_mid}, Bottom: {optimal_bottom}")
    
    # Get labels at each level
    top_labels    = fcluster(linkage_matrix, t=optimal_top,    criterion='maxclust')
    mid_labels    = fcluster(linkage_matrix, t=optimal_mid,    criterion='maxclust')
    bottom_labels = fcluster(linkage_matrix, t=optimal_bottom, criterion='maxclust')
    
    # Auto name clusters using TF-IDF centroid
    def name_cluster(aspect_list, opinion_list):
        # Combine aspects and opinions for richer naming
        doc = " ".join(aspect_list + opinion_list)
        vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english')
        try:
            tfidf = vectorizer.fit_transform([doc])
            top_idx = tfidf.toarray()[0].argmax()
            return vectorizer.get_feature_names_out()[top_idx].title()
        except:
            return aspect_list[0].title() if aspect_list else "Unknown"
    
    # Build the tree
    tree = {}
    
    for top_id in set(top_labels):
        top_indices = [i for i, l in enumerate(top_labels) if l == top_id]
        top_aspects = [aspects[i] for i in top_indices]
        top_opinions = aspect_summary.iloc[top_indices]['sample_opinions'].tolist()
        top_name = name_cluster(top_aspects, top_opinions)
        
        # Aggregate sentiment for this top node
        top_data = aspect_summary.iloc[top_indices]
        top_sentiment = top_data['dominant_sentiment'].mode()[0]
        top_category  = top_data['dominant_category'].mode()[0]
        top_mentions  = top_data['mention_count'].sum()
        
        tree[top_name] = {
            "sentiment":  top_sentiment,
            "category":   top_category,
            "mentions":   int(top_mentions),
            "aspects":    top_aspects,
            "children":   {}
        }
        
        # Mid level
        for mid_id in set(mid_labels[top_indices]):
            mid_indices = [i for i in top_indices if mid_labels[i] == mid_id]
            mid_aspects = [aspects[i] for i in mid_indices]
            mid_opinions = aspect_summary.iloc[mid_indices]['sample_opinions'].tolist()
            mid_name = name_cluster(mid_aspects, mid_opinions)
            
            mid_data = aspect_summary.iloc[mid_indices]
            mid_sentiment = mid_data['dominant_sentiment'].mode()[0]
            mid_category  = mid_data['dominant_category'].mode()[0]
            mid_mentions  = mid_data['mention_count'].sum()
            
            tree[top_name]["children"][mid_name] = {
                "sentiment": mid_sentiment,
                "category":  mid_category,
                "mentions":  int(mid_mentions),
                "aspects":   mid_aspects,
                "children":  {}
            }
            
            # Bottom level
            for bottom_id in set(bottom_labels[mid_indices]):
                bottom_indices = [i for i in mid_indices if bottom_labels[i] == bottom_id]
                bottom_aspects = [aspects[i] for i in bottom_indices]
                bottom_opinions = aspect_summary.iloc[bottom_indices]['sample_opinions'].tolist()
                bottom_name = name_cluster(bottom_aspects, bottom_opinions)
                
                bottom_data = aspect_summary.iloc[bottom_indices]
                bottom_sentiment = bottom_data['dominant_sentiment'].mode()[0]
                bottom_category  = bottom_data['dominant_category'].mode()[0]
                bottom_mentions  = bottom_data['mention_count'].sum()
                
                tree[top_name]["children"][mid_name]["children"][bottom_name] = {
                    "sentiment": bottom_sentiment,
                    "category":  bottom_category,
                    "mentions":  int(bottom_mentions),
                    "aspects":   bottom_aspects
                }
    
    return tree

def print_tree(tree, indent=0):
    """Pretty print the review tree"""
    for name, data in tree.items():
        prefix = "  " * indent + ("├── " if indent > 0 else "")
        print(f"{prefix}{name} | {data['category']} | {data['sentiment']} | {data['mentions']} mentions")
        if "children" in data and data["children"]:
            print_tree(data["children"], indent + 1)

# Run
if __name__ == "__main__":
    df_absa = pd.read_csv("absa_output_electric_tb.csv")
    tree = build_review_tree(df_absa)
    print("\n=== Review Tree ===\n")
    print_tree(tree)


          review_id  star_rating                                                                                                                                                                                                                                                                         sentence                  aspect   opinion_term sentiment  confidence   category
0    A2ROBO96G69ABZ            5                                                                                                                                                                                                                                                 be careful around the gum lines.               gum lines            NaN  Negative      0.9620      Worst
1    A2ROBO96G69ABZ            5                                                                                                                                                                                                                    

/opt/anaconda3/envs/myenv_DSL/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


RuntimeError: Numpy is not available

In [4]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
import torch

# ── Load embedding model ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
embed_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

def get_embeddings(texts):
    embeddings = []
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=128
        )
        with torch.no_grad():
            outputs = embed_model(**inputs)

        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).float()
        embedding = torch.sum(token_embeddings * input_mask_expanded, 1)
        embedding = embedding / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embeddings.append(embedding.squeeze().tolist())

    return np.array(embeddings)

# ── Category classifier ────────────────────────────────────────────────────────
def classify_category(sentiment, confidence):
    if sentiment == "Positive":
        if confidence >= 0.95:
            return "Excellent"
        elif confidence >= 0.75:
            return "Good"
        else:
            return "Average"
    elif sentiment == "Negative":
        if confidence >= 0.95:
            return "Worst"
        elif confidence >= 0.75:
            return "Bad"
        else:
            return "Average"
    else:
        return "Average"

# ── Cluster namer ──────────────────────────────────────────────────────────────
def name_cluster(aspect_list, opinion_list):
    doc = " ".join(aspect_list + opinion_list)
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
    try:
        tfidf = vectorizer.fit_transform([doc])
        top_idx = tfidf.toarray()[0].argmax()
        return vectorizer.get_feature_names_out()[top_idx].title()
    except:
        return aspect_list[0].title() if aspect_list else "Unknown"

# ── Tree builder ───────────────────────────────────────────────────────────────
def build_review_tree(df_absa):

    # Add category column if not present
    if 'category' not in df_absa.columns:
        df_absa['category'] = df_absa.apply(
            lambda row: classify_category(row['sentiment'], row['confidence']), axis=1
        )

    # Summarize aspects
    aspect_summary = df_absa.groupby('aspect').agg(
        avg_confidence      = ('confidence', 'mean'),
        dominant_sentiment  = ('sentiment',  lambda x: x.mode()[0]),
        dominant_category   = ('category',   lambda x: x.mode()[0]),
        mention_count       = ('aspect',     'count'),
        sample_opinions     = ('opinion_term', lambda x: ' '.join(x.dropna().unique()[:3]))
    ).reset_index()

    aspects = aspect_summary['aspect'].tolist()

    if len(aspects) < 3:
        print("Too few aspects to cluster")
        return {}

    print(f"Embedding {len(aspects)} aspects...")
    embeddings = get_embeddings(aspects)

    # Build linkage matrix
    linkage_matrix = linkage(embeddings, method='ward')

    # Grid search for optimal K at each level
    def find_optimal_k(k_range):
        scores = {}
        for k in k_range:
            if k >= len(aspects):
                continue
            labels = fcluster(linkage_matrix, t=k, criterion='maxclust')
            if len(set(labels)) < 2:
                continue
            score = silhouette_score(embeddings, labels)
            scores[k] = score
        return max(scores, key=scores.get) if scores else list(k_range)[0]

    optimal_top    = find_optimal_k(range(2, 6))
    optimal_mid    = find_optimal_k(range(optimal_top + 1, 12))
    optimal_bottom = find_optimal_k(range(optimal_mid + 1, 25))

    print(f"Optimal cuts — Top: {optimal_top}, Mid: {optimal_mid}, Bottom: {optimal_bottom}")

    top_labels    = fcluster(linkage_matrix, t=optimal_top,    criterion='maxclust')
    mid_labels    = fcluster(linkage_matrix, t=optimal_mid,    criterion='maxclust')
    bottom_labels = fcluster(linkage_matrix, t=optimal_bottom, criterion='maxclust')

    # Build tree
    tree = {}

    for top_id in set(top_labels):
        top_indices  = [i for i, l in enumerate(top_labels) if l == top_id]
        top_aspects  = [aspects[i] for i in top_indices]
        top_opinions = aspect_summary.iloc[top_indices]['sample_opinions'].tolist()
        top_name     = name_cluster(top_aspects, top_opinions)

        top_data      = aspect_summary.iloc[top_indices]
        top_sentiment = top_data['dominant_sentiment'].mode()[0]
        top_category  = top_data['dominant_category'].mode()[0]
        top_mentions  = int(top_data['mention_count'].sum())

        tree[top_name] = {
            "sentiment": top_sentiment,
            "category":  top_category,
            "mentions":  top_mentions,
            "aspects":   top_aspects,
            "children":  {}
        }

        for mid_id in set(mid_labels[i] for i in top_indices):
            mid_indices  = [i for i in top_indices if mid_labels[i] == mid_id]
            mid_aspects  = [aspects[i] for i in mid_indices]
            mid_opinions = aspect_summary.iloc[mid_indices]['sample_opinions'].tolist()
            mid_name     = name_cluster(mid_aspects, mid_opinions)

            mid_data      = aspect_summary.iloc[mid_indices]
            mid_sentiment = mid_data['dominant_sentiment'].mode()[0]
            mid_category  = mid_data['dominant_category'].mode()[0]
            mid_mentions  = int(mid_data['mention_count'].sum())

            tree[top_name]["children"][mid_name] = {
                "sentiment": mid_sentiment,
                "category":  mid_category,
                "mentions":  mid_mentions,
                "aspects":   mid_aspects,
                "children":  {}
            }

            for bottom_id in set(bottom_labels[i] for i in mid_indices):
                bottom_indices  = [i for i in mid_indices if bottom_labels[i] == bottom_id]
                bottom_aspects  = [aspects[i] for i in bottom_indices]
                bottom_opinions = aspect_summary.iloc[bottom_indices]['sample_opinions'].tolist()
                bottom_name     = name_cluster(bottom_aspects, bottom_opinions)

                bottom_data      = aspect_summary.iloc[bottom_indices]
                bottom_sentiment = bottom_data['dominant_sentiment'].mode()[0]
                bottom_category  = bottom_data['dominant_category'].mode()[0]
                bottom_mentions  = int(bottom_data['mention_count'].sum())

                tree[top_name]["children"][mid_name]["children"][bottom_name] = {
                    "sentiment": bottom_sentiment,
                    "category":  bottom_category,
                    "mentions":  bottom_mentions,
                    "aspects":   bottom_aspects
                }

    return tree

# ── Pretty print ───────────────────────────────────────────────────────────────
def print_tree(tree, indent=0):
    for name, data in tree.items():
        prefix = "  " * indent + ("├── " if indent > 0 else "")
        print(f"{prefix}{name} | {data['category']} | {data['sentiment']} | {data['mentions']} mentions")
        if "children" in data and data["children"]:
            print_tree(data["children"], indent + 1)

# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_absa = pd.read_csv("absa_output_electric_tb.csv")
    tree = build_review_tree(df_absa)
    print("\n=== Review Tree ===\n")
    print_tree(tree)

/opt/anaconda3/envs/myenv_DSL/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding 595 aspects...
Optimal cuts — Top: 2, Mid: 3, Bottom: 21

=== Review Tree ===

Brush | Average | Positive | 457 mentions
  ├── Brush | Average | Positive | 457 mentions
    ├── Brush | Average | Negative | 270 mentions
    ├── Clean | Average | Positive | 48 mentions
    ├── Sonicare | Average | Positive | 19 mentions
    ├── Oral | Average | Positive | 120 mentions
Charging | Average | Positive | 1049 mentions
  ├── Battery | Average | Negative | 198 mentions
    ├── Charging | Average | Negative | 76 mentions
    ├── Battery | Average | Negative | 61 mentions
    ├── Base | Average | Neutral | 61 mentions
  ├── Great | Average | Positive | 851 mentions
    ├── Timer | Average | Neutral | 26 mentions
    ├── Indicator | Average | Negative | 25 mentions
    ├── Sensitive | Average | Positive | 12 mentions
    ├── Whitening | Average | Positive | 13 mentions
    ├── Button | Average | Neutral | 59 mentions
    ├── Case | Average | Neutral | 38 mentions
    ├── Great | Average 

FINAL ADDING LEVELS

In [5]:
#ADDING CLUSTERS IN THE END

import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
import torch

# ── Load embedding model ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
embed_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

def get_embeddings(texts):
    embeddings = []
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=128
        )
        with torch.no_grad():
            outputs = embed_model(**inputs)

        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).float()
        embedding = torch.sum(token_embeddings * input_mask_expanded, 1)
        embedding = embedding / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embeddings.append(embedding.squeeze().tolist())

    return np.array(embeddings)

def name_cluster(aspect_list, opinion_list):
    doc = " ".join(aspect_list + opinion_list)
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
    try:
        tfidf = vectorizer.fit_transform([doc])
        top_idx = tfidf.toarray()[0].argmax()
        return vectorizer.get_feature_names_out()[top_idx].title()
    except:
        return aspect_list[0].title() if aspect_list else "Unknown"

def add_cluster_columns(df_absa):

    if 'category' not in df_absa.columns:
        def classify_category(sentiment, confidence):
            if sentiment == "Positive":
                if confidence >= 0.95:   return "Excellent"
                elif confidence >= 0.75: return "Good"
                else:                    return "Average"
            elif sentiment == "Negative":
                if confidence >= 0.95:   return "Worst"
                elif confidence >= 0.75: return "Bad"
                else:                    return "Average"
            else:
                return "Average"
        df_absa['category'] = df_absa.apply(
            lambda row: classify_category(row['sentiment'], row['confidence']), axis=1
        )

    # Summarize aspects
    aspect_summary = df_absa.groupby('aspect').agg(
        dominant_sentiment = ('sentiment',    lambda x: x.mode()[0]),
        dominant_category  = ('category',     lambda x: x.mode()[0]),
        mention_count      = ('aspect',       'count'),
        sample_opinions    = ('opinion_term', lambda x: ' '.join(x.dropna().unique()[:3]))
    ).reset_index()

    aspects = aspect_summary['aspect'].tolist()

    print(f"Embedding {len(aspects)} aspects...")
    embeddings = get_embeddings(aspects)

    # Build linkage matrix
    linkage_matrix = linkage(embeddings, method='ward')

    # Grid search
    def find_optimal_k(k_range):
        scores = {}
        for k in k_range:
            if k >= len(aspects):
                continue
            labels = fcluster(linkage_matrix, t=k, criterion='maxclust')
            if len(set(labels)) < 2:
                continue
            score = silhouette_score(embeddings, labels)
            scores[k] = score
        return max(scores, key=scores.get) if scores else list(k_range)[0]

    optimal_top = find_optimal_k(range(2, 6))
    optimal_mid = find_optimal_k(range(optimal_top + 1, 12))

    print(f"Optimal cuts — Top: {optimal_top}, Mid: {optimal_mid}")

    top_labels = fcluster(linkage_matrix, t=optimal_top, criterion='maxclust')
    mid_labels = fcluster(linkage_matrix, t=optimal_mid, criterion='maxclust')

    # Build aspect -> cluster name mapping
    top_cluster_map = {}
    mid_cluster_map = {}

    for top_id in set(top_labels):
        top_indices  = [i for i, l in enumerate(top_labels) if l == top_id]
        top_aspects  = [aspects[i] for i in top_indices]
        top_opinions = aspect_summary.iloc[top_indices]['sample_opinions'].tolist()
        top_name     = name_cluster(top_aspects, top_opinions)

        for i in top_indices:
            top_cluster_map[aspects[i]] = top_name

    for mid_id in set(mid_labels):
        mid_indices  = [i for i, l in enumerate(mid_labels) if l == mid_id]
        mid_aspects  = [aspects[i] for i in mid_indices]
        mid_opinions = aspect_summary.iloc[mid_indices]['sample_opinions'].tolist()
        mid_name     = name_cluster(mid_aspects, mid_opinions)

        for i in mid_indices:
            mid_cluster_map[aspects[i]] = mid_name

    # Map back to original dataframe
    df_absa['cluster_level_top']    = df_absa['aspect'].map(top_cluster_map)
    df_absa['cluster_level_bottom'] = df_absa['aspect'].map(mid_cluster_map)

    return df_absa

# ── Run ────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_absa = pd.read_csv("final_bucket_output (5).csv")
    
    # Save original columns and data
    original_df = df_absa.copy()
    print("Original columns:", original_df.columns.tolist())
    
    # Run clustering - only to get the two new columns
    df_with_clusters = add_cluster_columns(df_absa.copy())
    
    # Extract only the two new cluster columns
    original_df['cluster_level_top']    = df_with_clusters['cluster_level_top']
    original_df['cluster_level_bottom'] = df_with_clusters['cluster_level_bottom']
    
    print("Final columns:", original_df.columns.tolist())
    print("\nSample:")
    print(original_df.head(10).to_string())
    
    original_df.to_csv("absa_output_clustered_real.csv", index=False)
    print("\nSaved to absa_output_clustered_real.csv")

/opt/anaconda3/envs/myenv_DSL/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Original columns: ['review_id', 'Brand', 'star_rating', 'sentence', 'aspect', 'opinion_term', 'sentiment', 'confidence', 'text_for_analysis', 'sbert_bucket', 'w2v_bucket', 'glove_bucket', 'Unnamed: 12']
Embedding 595 aspects...
Optimal cuts — Top: 2, Mid: 3
Final columns: ['review_id', 'Brand', 'star_rating', 'sentence', 'aspect', 'opinion_term', 'sentiment', 'confidence', 'text_for_analysis', 'sbert_bucket', 'w2v_bucket', 'glove_bucket', 'Unnamed: 12', 'cluster_level_top', 'cluster_level_bottom']

Sample:
        review_id       Brand  star_rating                                                                                                                                       sentence      aspect opinion_term sentiment  confidence text_for_analysis sbert_bucket w2v_bucket glove_bucket  Unnamed: 12 cluster_level_top cluster_level_bottom
0  A2ROBO96G69ABZ  B007ZN5ATQ            5                                                                                                          

In [3]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic.representation import KeyBERTInspired

# Explicitly define embedding model so BERTopic does not try to build its own
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    min_topic_size=2,
    nr_topics="auto",
    verbose=True
)

NameError: name 'nn' is not defined

In [ ]:
# Trying Betropic


from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

def build_tree_bertopic(df_absa):
    
    # Clean first
    df_clean = clean_aspects(df_absa)
    
    aspect_summary = df_clean.groupby('aspect').agg(
        dominant_sentiment = ('sentiment',    lambda x: x.mode()[0]),
        dominant_category  = ('category',     lambda x: x.mode()[0]),
        mention_count      = ('aspect',       'count'),
        sample_opinions    = ('opinion_term', lambda x: ' '.join(x.dropna().unique()[:5]))
    ).reset_index()
    
    aspects = aspect_summary['aspect'].tolist()
    
    if len(aspects) < 5:
        print("Too few aspects after filtering")
        return {}
    
    # Combine aspect with its opinion terms for richer representation
    docs = [
        f"{row['aspect']} {row['sample_opinions']}"
        for _, row in aspect_summary.iterrows()
    ]
    
    # BERTopic with automatic topic discovery
    # min_topic_size controls minimum cluster size - adjust based on your data
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2))
    
    topic_model = BERTopic(
        vectorizer_model=vectorizer,
        min_topic_size=2,
        nr_topics="auto",
        verbose=True
    )
    
    topics, probs = topic_model.fit_transform(docs)
    
    # Get topic info with auto generated names
    topic_info = topic_model.get_topic_info()
    print("\nAuto discovered topics:")
    print(topic_info.to_string())
    
    # Build tree from topics
    tree = {}
    
    for topic_id in set(topics):
        if topic_id == -1:
            # These are outliers BERTopic could not assign
            # Put them in an Other category
            indices = [i for i, t in enumerate(topics) if t == -1]
            if indices:
                tree["Other"] = {
                    "sentiment": "Neutral",
                    "category":  "Average",
                    "mentions":  int(sum(aspect_summary.iloc[i]['mention_count'] for i in indices)),
                    "aspects":   [aspects[i] for i in indices],
                    "children":  {}
                }
            continue
        
        indices = [i for i, t in enumerate(topics) if t == topic_id]
        topic_aspects = [aspects[i] for i in indices]
        
        # Get auto name from BERTopic
        topic_words = topic_model.get_topic(topic_id)
        topic_name  = " ".join([w for w, _ in topic_words[:2]]).title()
        
        topic_data    = aspect_summary.iloc[indices]
        top_sentiment = topic_data['dominant_sentiment'].mode()[0]
        top_category  = topic_data['dominant_category'].mode()[0]
        top_mentions  = int(topic_data['mention_count'].sum())
        
        tree[topic_name] = {
            "sentiment": top_sentiment,
            "category":  top_category,
            "mentions":  top_mentions,
            "aspects":   topic_aspects,
            "children":  {}
        }
        
        # Second level within each topic using HAC
        if len(indices) >= 4:
            sub_docs = [docs[i] for i in indices]
            sub_embeddings = get_embeddings(sub_docs)
            
            if len(sub_embeddings) >= 4:
                from scipy.cluster.hierarchy import linkage, fcluster
                from sklearn.metrics import silhouette_score
                
                linkage_matrix = linkage(sub_embeddings, method='ward')
                
                best_k = 2
                best_score = -1
                for k in range(2, min(len(indices), 6)):
                    labels = fcluster(linkage_matrix, t=k, criterion='maxclust')
                    if len(set(labels)) < 2:
                        continue
                    score = silhouette_score(sub_embeddings, labels)
                    if score > best_score:
                        best_score = score
                        best_k = k
                
                sub_labels = fcluster(linkage_matrix, t=best_k, criterion='maxclust')
                
                for sub_id in set(sub_labels):
                    sub_indices  = [indices[i] for i, l in enumerate(sub_labels) if l == sub_id]
                    sub_aspects  = [aspects[i] for i in sub_indices]
                    sub_data     = aspect_summary.iloc[sub_indices]
                    sub_name     = sub_aspects[0].title() if sub_aspects else "Unknown"
                    sub_sent     = sub_data['dominant_sentiment'].mode()[0]
                    sub_cat      = sub_data['dominant_category'].mode()[0]
                    sub_mentions = int(sub_data['mention_count'].sum())
                    
                    tree[topic_name]["children"][sub_name] = {
                        "sentiment": sub_sent,
                        "category":  sub_cat,
                        "mentions":  sub_mentions,
                        "aspects":   sub_aspects
                    }
    
    return tree

def print_tree(tree, indent=0):
    for name, data in tree.items():
        prefix = "  " * indent + ("├── " if indent > 0 else "")
        print(f"{prefix}{name} | {data['category']} | {data['sentiment']} | {data['mentions']} mentions")
        if "children" in data and data["children"]:
            print_tree(data["children"], indent + 1)

if __name__ == "__main__":
    df_absa = pd.read_csv("final_bucket_output (5).csv")
    tree = build_tree_bertopic(df_absa)
    print("\n=== Review Tree ===\n")
    print_tree(tree)


/opt/anaconda3/envs/myenv_DSL/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NameError: name 'nn' is not defined